# 00 — Setup & Data Download (FathomNetCLEF2026)

**Run this first.** Its output (`/kaggle/working`) becomes the *Input* of the
EDA / training / eval notebooks.

⚠️ **Settings → Internet must be ON** (images are fetched from FathomNet's servers).
This notebook falls back to a built-in threaded downloader if the official
`download.py` dependencies (`coco-lib`) can't be pip-installed.

**Disk note (measured):** FathomNet serves full-res PNGs at ~6.5 MB/image -> the raw set is ~50 GB, 2.5x Kaggle's 19.5 GB working cap. The official download.py therefore cannot finish here. Cell 2 streams every image to JPEG q=90 in memory (peak disk == final size: ~8-12 GB full-res, ~4-6 GB if RESIZE_MAX_SIDE=1280), auto-converts PNGs left by earlier runs, and updates both JSONs (file_name; width/height/bboxes only if you resize). Run cells 0-4 in order, re-running cell 2 once if a few downloads failed, then Save Version.

In [1]:
# --- 0. Install downloader dependencies (run FIRST) ----------------------
import subprocess
from pathlib import Path
COMP = Path('/kaggle/input/competitions/fathomnet-2026')
WORK = Path('/kaggle/working')

# Try the OFFICIAL downloader (preferred) ---------------------------------
import subprocess

official_ok = True
try:
    # tqdm installed explicitly: required by the official download.py
    # (from tqdm.asyncio import tqdm) even when requirements.txt fails
    subprocess.run(
        ['pip', 'install', '-q', 'tqdm>=4.67', '-r', str(COMP / 'requirements.txt')],
        check=True, capture_output=True, timeout=300)
    import coco_lib, httpx, tenacity  # noqa: F401
    from tqdm.asyncio import tqdm  # verify download.py's import works too
    print('✅ Official download.py dependencies installed (incl. tqdm)')
except Exception as e:
    official_ok = False
    print(f'⚠️ Official deps unavailable ({type(e).__name__}) - will use built-in fallback downloader')

✅ Official download.py dependencies installed (incl. tqdm)


In [2]:
# --- 1. Check connectivity BEFORE anything else -------------------------
import requests
from pathlib import Path
import shutil

COMP = Path('/kaggle/input/competitions/fathomnet-2026')
WORK = Path('/kaggle/working')
print('Competition files:', sorted(p.name for p in COMP.glob('*')))

try:
    requests.head('https://www.kaggle.com', timeout=10)
    print('✅ Internet is ON')
except requests.ConnectionError:
    raise RuntimeError(
        '❌ NO INTERNET. Go to Settings (right panel) -> Internet -> ON, '
        'and add a phone verification if Kaggle asks for it, then rerun this notebook.'
    )

# Copy annotation JSONs into working dir (so outputs are linkable later)
for name in ['train_dataset.json', 'test_dataset.json']:
    shutil.copy(COMP / name, WORK / name)
print('Copied JSONs ->', WORK)

Competition files: ['README.md', 'download.py', 'requirements.txt', 'test_dataset.json', 'train_dataset.json']
✅ Internet is ON
Copied JSONs -> /kaggle/working


In [3]:
# --- 2. Download images (streaming PNG->JPEG, disk-safe) ------------------
# MEASURED from your run: ~6.5 MB/image -> full raw set ~50 GB, i.e. 2.5x
# Kaggle's 19.5 GB /kaggle/working cap. The official download.py therefore
# ALWAYS dies mid-run with disk-full, no matter how many times you resume.
# This cell instead downloads each image, converts to JPEG q=90 IN MEMORY,
# and writes only the JPEG to disk: peak usage == final library size.
# Existing PNGs (from the official-script run) are converted first.
import io, json, time, random
import concurrent.futures as cf
from pathlib import Path
import requests
from PIL import Image
from tqdm.auto import tqdm

IMG = WORK / 'images'
IMG.mkdir(parents=True, exist_ok=True)
JPEG_QUALITY = 90
RESIZE_MAX_SIDE = None    # e.g. 1280 -> also downscale; JSONs updated below
WORKERS = 12              # 24 workers triggers server disconnects; 12 is stable

def gb(path):
    return sum(f.stat().st_size for f in path.rglob('*') if f.is_file()) / 1e9

def to_jpeg_bytes(content):
    pic = Image.open(io.BytesIO(content)).convert('RGB')
    if RESIZE_MAX_SIDE:
        scale = RESIZE_MAX_SIDE / max(pic.size)
        if scale < 1:
            pic = pic.resize((round(pic.width * scale), round(pic.height * scale)),
                             Image.LANCZOS)
    buf = io.BytesIO()
    pic.save(buf, 'JPEG', quality=JPEG_QUALITY, optimize=True)
    return buf.getvalue()

def fetch(url):
    for attempt in range(5):  # retry w/ backoff: survives transient disconnects
        try:
            r = requests.get(url, timeout=90)
            r.raise_for_status()
            return r.content
        except Exception:
            time.sleep(min(2 ** attempt, 30) + random.random())
    return None

def jpg_name(fname):
    return Path(fname).with_suffix('.jpg').name

def fix_json(jname):
    """Point file_name at .jpg (+rescale dims/bboxes if RESIZE_MAX_SIDE)."""
    coco = json.load(open(WORK / jname))
    anns = {}
    for a in coco.get('annotations', []):
        anns.setdefault(a['image_id'], []).append(a)
    linked = 0
    for im in coco['images']:
        if not im.get('coco_url'):
            continue
        p = IMG / jpg_name(im['file_name'])
        if not p.exists():
            continue
        linked += 1
        im['file_name'] = p.name
        if RESIZE_MAX_SIDE and max(im['width'], im['height']) > RESIZE_MAX_SIDE:
            scale = RESIZE_MAX_SIDE / max(im['width'], im['height'])
            im['width'] = round(im['width'] * scale)
            im['height'] = round(im['height'] * scale)
            for a in anns.get(im['id'], []):
                x, y, w, h = a['bbox']
                a['bbox'] = [x * scale, y * scale, w * scale, h * scale]
    json.dump(coco, open(WORK / jname, 'w'))
    return linked

if IMG.exists():
    print(f'size before: {gb(IMG):.2f} GB')

pngs = list(IMG.glob('*.png')) if IMG.exists() else []
if pngs:
    print(f'converting {len(pngs)} existing PNGs -> JPEG ...')
    ok0 = 0
    for src in tqdm(pngs, unit='img'):
        dst = src.with_suffix('.jpg')
        try:
            dst.write_bytes(to_jpeg_bytes(src.read_bytes()))
            if dst.stat().st_size < src.stat().st_size:
                src.unlink()
                ok0 += 1
            else:
                dst.unlink()
        except Exception:
            dst.unlink(missing_ok=True)
    print(f'converted {ok0}/{len(pngs)}')
for leftover in IMG.glob('*.part'):
    leftover.unlink()

for jname in ['train_dataset.json', 'test_dataset.json']:
    coco = json.load(open(WORK / jname))
    todo, have = [], 0
    for im in coco['images']:
        if not im.get('coco_url'):
            continue
        p = IMG / jpg_name(im['file_name'])
        if p.exists() and p.stat().st_size > 0:
            have += 1
        else:
            todo.append((im['coco_url'], p))
    print(f'{jname}: {have} already done, {len(todo)} to download')

    def grab(task):
        url, p = task
        raw = fetch(url)
        if raw is None:
            return False
        try:
            data = to_jpeg_bytes(raw)
        except Exception:
            return False
        tmp = p.with_suffix('.part')
        tmp.write_bytes(data)
        tmp.rename(p)
        return True

    t0 = time.time()
    with cf.ThreadPoolExecutor(max_workers=WORKERS) as ex:
        res = list(tqdm(ex.map(grab, todo), total=len(todo),
                        desc=f'downloading {jname}', unit='img'))
    rate = (time.time() - t0) / 60
    print(f'{sum(res)}/{len(todo)} succeeded in {rate:.1f} min'
          + ('' if all(res) else '  -> re-run this cell to retry the failures'))
    print(f'json linked to jpg: {fix_json(jname)} images')

if IMG.exists():
    print(f'size after : {gb(IMG):.2f} GB  (peak disk == this; PNGs never stored)')


size before: 0.00 GB
train_dataset.json: 0 already done, 6463 to download


downloading train_dataset.json:   0%|          | 0/6463 [00:00<?, ?img/s]

6463/6463 succeeded in 17.1 min
json linked to jpg: 6463 images
test_dataset.json: 0 already done, 1425 to download


downloading test_dataset.json:   0%|          | 0/1425 [00:00<?, ?img/s]

1425/1425 succeeded in 2.7 min
json linked to jpg: 1425 images
size after : 6.69 GB  (peak disk == this; PNGs never stored)


In [4]:
# --- 3. Compress PNG -> JPEG (safety net; cell 2 already streams to JPEG) --
# Only does work if PNGs are on disk (e.g. you ran the official download.py
# before). Photographic ROV PNGs shrink 4-6x as JPEG q=90; resolution is kept,
# so bboxes stay valid. Set RESIZE_MAX_SIDE to downscale + rescale bboxes.
import json
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

COMPRESS = True
RESIZE_MAX_SIDE = None
JPEG_QUALITY = 90

def gb(p):
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e9

IMG = Path('/kaggle/working/images')
for part in IMG.glob('*.part'):
    part.unlink()

if COMPRESS and IMG.exists():
    print(f'size before: {gb(IMG):.2f} GB')
    for jname in ['train_dataset.json', 'test_dataset.json']:
        coco = json.load(open(WORK / jname))
        anns_by_img = {}
        for a in coco.get('annotations', []):
            anns_by_img.setdefault(a['image_id'], []).append(a)
        converted = resized_n = 0
        for im in tqdm(coco['images'], desc=f'compress {jname}', unit='img'):
            src = IMG / im['file_name']
            if not src.exists() or src.suffix.lower() == '.jpg':
                continue
            scale = 1.0
            with Image.open(src) as pic:
                pic = pic.convert('RGB')
                if RESIZE_MAX_SIDE:
                    scale = RESIZE_MAX_SIDE / max(pic.size)
                    if scale < 1:
                        pic = pic.resize(
                            (round(pic.width * scale), round(pic.height * scale)),
                            Image.LANCZOS)
                dst = src.with_suffix('.jpg')
                pic.save(dst, 'JPEG', quality=JPEG_QUALITY, optimize=True)
            if dst.stat().st_size < src.stat().st_size:
                src.unlink()
                converted += 1
                im['file_name'] = dst.name
                if scale < 1:
                    resized_n += 1
                    im['width'] = round(im['width'] * scale)
                    im['height'] = round(im['height'] * scale)
                    for a in anns_by_img.get(im['id'], []):
                        x, y, w, h = a['bbox']
                        a['bbox'] = [x * scale, y * scale, w * scale, h * scale]
            else:
                dst.unlink()
        json.dump(coco, open(WORK / jname, 'w'))
        print(f'{jname}: {converted} converted, {resized_n} resized')
    print(f'size after : {gb(IMG):.2f} GB')
else:
    print('nothing to compress (no PNGs on disk or COMPRESS=False)')


size before: 6.69 GB


compress train_dataset.json:   0%|          | 0/6463 [00:00<?, ?img/s]

train_dataset.json: 0 converted, 0 resized


compress test_dataset.json:   0%|          | 0/1425 [00:00<?, ?img/s]

test_dataset.json: 0 converted, 0 resized
size after : 6.69 GB


In [5]:
# --- 4. Sanity check: images vs JSONs ------------------------------------
n_train = len(json.load(open(WORK / 'train_dataset.json'))['images'])
n_test = len(json.load(open(WORK / 'test_dataset.json'))['images'])
imgs = [p for p in IMG.glob('*') if p.suffix.lower() in ('.png', '.jpg', '.jpeg')]
print(f'Expected: {n_train} train + {n_test} test = {n_train + n_test}')
print(f'Found   : {len(imgs)} images in {IMG}')
if len(imgs) < (n_train + n_test) * 0.98:
    print('⚠️ Missing images - re-run the download cell (it resumes, skipping existing files).')
else:
    print('✅ Dataset ready. Now SAVE VERSION, then use this output as Input for notebooks 01-04.')

Expected: 6463 train + 1425 test = 7888
Found   : 7888 images in /kaggle/working/images
✅ Dataset ready. Now SAVE VERSION, then use this output as Input for notebooks 01-04.
